# Persian STT — Noisy-Condition Robustness Benchmark

Round-based evaluation of the **10 production-candidate models** (clean-FLEURS WER ≤ 0.30) under noisy / real-world conditions, for a clinical (doctor–patient) Persian STT deployment.

**Pick a dataset at the top (`DATASET = ...`) and Run All.** Each round writes its own tagged folder of CSVs under `/kaggle/working/<dataset>/`, so rounds never overwrite each other.

| Round | `DATASET` | What it tests |
| --- | --- | --- |
| 1 | `psrb` | Real noisy + spontaneous Persian (PartAI/PSRB, non-clean subset) |
| – | `common_voice_fa` | Real read speech, varied mics (HuggingFace — **gated**) |
| 3 | `synthetic` | FLEURS clips + additive Gaussian noise at fixed SNRs (paired → ΔWER) |
| 3a | `synthetic_realistic` | **Clinic-realistic:** room reverb + cheap-mic chain + Gaussian, swept over SNR (paired → ΔWER) |
| 3b | `synthetic_worst` | **Worst-case clinic:** reverb + babble + cough + HVAC hum + monitor beeps + Gaussian + mic, swept over SNR (paired → ΔWER) |
| base | `fleurs_clean` | Clean FLEURS fa_ir baseline + ΔWER reference |

> Rounds **3a / 3b** reuse the noise-playground engine. Attach **MUSAN** + **ESC-50** (and optionally **DEMAND**) via *Add Data* for `synthetic_worst`; `synthetic_realistic` needs no banks.

**Models (10):** seamless-m4t-v2-large · hf-seamless-m4t-medium · whisper-persian-v4 · mms-1b-fl102 · whisper-large-fa-v1 · mms-1b-all · persian-whisper-large-v3 · whisper-large-v3 · whisper-large-v3-turbo · wav2vec2-large-xlsr-53-persian

**Metrics** — identical suite to the clean benchmark, plus noisy-round additions:
- *carried over:* WER · CER · chrF · BERTScore · semantic similarity · SER · WER P50/P90/P95 · %perfect · %catastrophic · sub/ins/del · hallucination ratio · script contamination · repetition · punct F1 · RTF · throughput · VRAM · WER-by-duration
- *added:* `snr_db` / `noise_type` (synthetic) · `acoustic_environment` / `spontaneous` (PSRB) · `delta_wer` (paired) · `is_hallucination` → **% hallucinated** · **bootstrap 95% CI** on corpus WER · `no_speech_prob` hook

> **Reading the noisy results:** lead with `wer_p50`, `pct_catastrophic`, and **% hallucinated** (with the WER CI). Under noise the mean WER is dominated by a few collapsed clips. chrF / BERTScore / semantic-sim stay deceptively high on partial matches — keep them as context, not pass/fail.

Install required Python packages (identical to the clean benchmark).

In [ ]:
!pip install -q jiwer hazm sacrebleu sentencepiece pyroomacoustics
!pip install -q --no-deps bert-score sentence-transformers

Import libraries and verify the GPU is available.

In [2]:
# ════════════════════════════════════════════════════════════════
#  IMPORTS
# ════════════════════════════════════════════════════════════════
import os, gc, re, time, io, warnings
from collections import Counter
import numpy as np
import pandas as pd
import soundfile as sf
import torch
import torchaudio
from datasets import load_dataset, load_from_disk, Audio
from transformers import (
    WhisperForConditionalGeneration, WhisperProcessor,
    Wav2Vec2ForCTC, Wav2Vec2Processor,
    SeamlessM4TModel, SeamlessM4Tv2Model, AutoProcessor,
)
from jiwer import wer, cer, process_words
from sacrebleu.metrics import CHRF
import hazm

warnings.filterwarnings("ignore")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {device}")
if device == "cuda":
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Device : cuda
GPU    : Tesla T4
VRAM   : 15.6 GB


**Select the round here.** Set `DATASET`, then Run All. Outputs are written to a per-dataset folder so rounds stay separate and comparable.

In [ ]:
# ════════════════════════════════════════════════════════════════
#  ROUND / DATASET SELECTION  —  set DATASET, then Run All
# ════════════════════════════════════════════════════════════════
#   "psrb"            Round 1 — PartAI/PSRB real noisy+spontaneous (HF, needs internet)
#   "fleurs_clean"    Clean FLEURS fa_ir baseline (also the delta_wer reference)
#   "common_voice_fa" Real read speech, varied mics (HF — GATED: token + accept terms)
#   "synthetic"            Round 3  — FLEURS clips + additive Gaussian noise at fixed SNRs
#   "synthetic_realistic"  Round 3a — clinic-realistic: reverb + cheap-mic + Gaussian (SNR sweep)
#   "synthetic_worst"      Round 3b — worst-case clinic: all noises combined (SNR sweep)
DATASET = "synthetic"

RESULTS_DIR = "/kaggle/working"
RUN_TAG     = DATASET
OUT_DIR     = os.path.join(RESULTS_DIR, RUN_TAG)
os.makedirs(OUT_DIR, exist_ok=True)

# ── FLEURS (clean baseline / synthetic source) ───────────────────
FLEURS_DISK_PATH = "/kaggle/input/Fleurs_fa_ir_test/fleurs_fa_ir_test"

# ── PSRB ─────────────────────────────────────────────────────────
PSRB_HF_ID            = "PartAI/PSRB"
PSRB_SPLIT            = "train"
PSRB_DROP_CLEAN       = True
PSRB_SPONTANEOUS_ONLY = False

# ── Common Voice ──────────────────────────────────────────────────
CV_HF_CANDIDATES = [
    "hezarai/common-voice-13-fa",             # Community mirror, no gating
    "mozilla-foundation/common_voice_17_0",   # Gated — needs HF_TOKEN + terms
    "mozilla-foundation/common_voice_16_1",   # Retired
]
CV_CONFIG = "default"   # hezarai mirror uses "default"; mozilla ones use "fa"
CV_SPLIT      = "test"
CV_MAX_CLIPS  = 500   # set to None to use the full split

# Set HF_TOKEN as a Kaggle secret (Add-ons -> Secrets) -- do NOT paste the token here
# (get it from huggingface.co → Settings → Access Tokens, role: read)
CV_HF_TOKEN = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN") or None

# ── Synthetic (Round 3 / 3a / 3b) ─────────────────────────────────
SYNTH_SNR_DB    = [20, 15, 10, 5, 0]   # SNR sweep (dB), shared by all synthetic rounds
SYNTH_NOISE     = "gaussian"
SYNTH_MAX_CLIPS = 200                   # FLEURS clips per SNR (None = all 871)
# Clinic profiles (synthetic_realistic / synthetic_worst) are defined in the NOISE
# ENGINE cell below — edit REALISTIC_PIPELINE / WORST_PIPELINE / PROFILE_REF_SNR there.
# synthetic_worst needs MUSAN + ESC-50 attached (babble + cough); they no-op if missing.

PAIRED_WITH_CLEAN  = DATASET.startswith("synthetic")
CLEAN_BASELINE_DIR = os.path.join(RESULTS_DIR, "fleurs_clean")

WER_CI_BOOTSTRAP = 1000
WER_CI_ALPHA     = 0.05

print(f"DATASET        : {DATASET}")
print(f"Output folder  : {OUT_DIR}")
print(f"Paired (ΔWER)  : {PAIRED_WITH_CLEAN}")

### Synthetic clinic-noise engine
Effect functions + noise banks (shared with the noise playground) and the two clinic profiles. Used only by `synthetic_realistic` / `synthetic_worst`. The SNR sweep (`SYNTH_SNR_DB`) shifts the whole additive-noise floor: at `PROFILE_REF_SNR` the mix equals the tuned params below, higher = cleaner, lower = louder. Reverb and the mic chain are channel effects, applied at fixed strength.

In [ ]:
# ════════════════════════════════════════════════════════════════
#  SYNTHETIC CLINIC-NOISE ENGINE  (shared with the noise playground)
#  Used only by DATASET = "synthetic_realistic" / "synthetic_worst".
# ════════════════════════════════════════════════════════════════
import csv, glob, subprocess, tempfile
from math import gcd
from scipy.signal import resample_poly, butter, sosfilt

SYNTH_SR        = 16000      # synthetic noise is mixed at 16 kHz
PROFILE_REF_SNR = 10         # sweep midpoint: at this SNR the pipeline == the tuned clinic mix

def _rms(x):
    return float(np.sqrt(np.mean(np.asarray(x, np.float64) ** 2)) + 1e-12)

def scale_to_snr(speech, noise, snr_db):
    target = _rms(speech) / (10.0 ** (snr_db / 20.0))
    return (noise * (target / _rms(noise))).astype(np.float32)

def _fit(x, n):
    if len(x) == 0:
        return np.zeros(n, np.float32)
    if len(x) < n:
        x = np.tile(x, int(np.ceil(n / len(x))))
    return x[:n].astype(np.float32)

def _load_wav(path, sr_target=SYNTH_SR):
    x, sr = sf.read(path, dtype="float32")
    if x.ndim > 1:
        x = x.mean(axis=1).astype(np.float32)
    if sr != sr_target and len(x):
        g = gcd(int(sr), int(sr_target))
        x = resample_poly(x, sr_target // g, sr // g).astype(np.float32)
    return x

# ── noise banks (MUSAN / ESC-50 / DEMAND), discovered under /kaggle/input ──
class NoiseBank:
    def __init__(self, files, sr):
        self.files = list(files); self.sr = sr
    def _read(self, path):
        try:
            return _load_wav(path, self.sr)
        except Exception:
            return np.zeros(0, np.float32)
    def sample(self, n, rng):
        if not self.files:
            return np.zeros(n, np.float32)
        x = self._read(str(rng.choice(self.files)))
        if len(x) == 0:
            return np.zeros(n, np.float32)
        if len(x) > n:
            s = int(rng.integers(0, len(x) - n + 1)); x = x[s:s + n]
        return _fit(x, n)
    def event(self, rng):
        if not self.files:
            return np.zeros(0, np.float32)
        return self._read(str(rng.choice(self.files)))

def _walk_wavs(root, exts=(".wav", ".flac", ".mp3", ".ogg")):
    if not root or not os.path.isdir(root):
        return []
    return [os.path.join(dp, f) for dp, _, fns in os.walk(root)
            for f in fns if f.lower().endswith(exts)]

def _find_dir(*keys, base="/kaggle/input"):
    for dp, dns, _ in os.walk(base):
        for d in dns:
            if any(k in d.lower() for k in keys):
                return os.path.join(dp, d)
    return None

def _subdir(root, name):
    if not root:
        return None
    for dp, dns, _ in os.walk(root):
        for d in dns:
            if d.lower() == name:
                return os.path.join(dp, d)
    return root

def _esc50_bank(root, categories, sr):
    cat = {}
    for dp, _, fns in os.walk(root or ""):
        for f in fns:
            if f.lower().endswith(".csv"):
                try:
                    for r in csv.DictReader(open(os.path.join(dp, f), encoding="utf-8")):
                        if "filename" in r and "category" in r:
                            cat[r["filename"]] = r["category"]
                except Exception:
                    pass
    allw = _walk_wavs(root)
    if cat and categories:
        files = [w for w in allw if cat.get(os.path.basename(w)) in set(categories)]
    else:
        files = allw
    return NoiseBank(files, sr)

_musan  = _find_dir("musan")
_esc    = _find_dir("esc50", "esc-50", "environmental-sound")
_demand = _find_dir("demand")
BANKS = {
    "babble":  NoiseBank(_walk_wavs(_subdir(_musan, "speech")), SYNTH_SR),
    "ambient": NoiseBank(_walk_wavs(_subdir(_musan, "noise")) + _walk_wavs(_demand), SYNTH_SR),
    "cough":   _esc50_bank(_esc, ["coughing", "breathing", "sneezing"], SYNTH_SR),
}
print("Noise banks:", {k: len(v.files) for k, v in BANKS.items()},
      f"(musan={_musan is not None}, esc50={_esc is not None}, demand={_demand is not None})")

# ── effect functions: fn(audio, sr, rng, **params) -> audio ──────
def reverb_synth(audio, sr, rng, rt60=0.4, **k):
    n = max(1, int(sr * rt60)); t = np.arange(n)
    ir = (rng.standard_normal(n) * np.exp(-6.908 * t / (rt60 * sr))).astype(np.float32)
    ir[0] += 1.0
    out = np.convolve(audio, ir)[:len(audio)]
    return (out * (_rms(audio) / _rms(out))).astype(np.float32)

def reverb_pyroom(audio, sr, rng, rt60=0.4, room=(4.0, 5.0, 3.0), **k):
    try:
        import pyroomacoustics as pra
        e_abs, max_order = pra.inverse_sabine(rt60, list(room))
        r = pra.ShoeBox(list(room), fs=sr, materials=pra.Material(e_abs), max_order=int(max_order))
        r.add_source([room[0] * 0.5, room[1] * 0.35, 1.2], signal=audio.astype(np.float64))
        r.add_microphone(np.array([room[0] * 0.5, room[1] * 0.65, 1.2]).reshape(3, 1))
        r.simulate()
        out = _fit(np.asarray(r.mic_array.signals[0], np.float32)[:len(audio)], len(audio))
        return (out * (_rms(audio) / _rms(out))).astype(np.float32)
    except Exception as e:
        print(f"    [reverb_pyroom -> synth fallback: {e}]")
        return reverb_synth(audio, sr, rng, rt60=rt60)

def add_gaussian(audio, sr, rng, snr_db=10, **k):
    p_sig = float(np.mean(audio.astype(np.float64) ** 2)) + 1e-12
    noise = rng.normal(0.0, np.sqrt(p_sig / (10.0 ** (snr_db / 10.0))), size=audio.shape)
    return (audio + noise.astype(np.float32)).astype(np.float32)

def add_noise(audio, sr, rng, bank="ambient", snr_db=15, **k):
    nb = BANKS.get(bank)
    if not nb or not nb.files:
        return audio
    return (audio + scale_to_snr(audio, nb.sample(len(audio), rng), snr_db)).astype(np.float32)

def add_babble(audio, sr, rng, snr_db=15, n_voices=4, bank="babble", **k):
    nb = BANKS.get(bank)
    if not nb or not nb.files:
        return audio
    mix = np.zeros(len(audio), np.float32)
    for _ in range(n_voices):
        mix += nb.sample(len(audio), rng)
    return (audio + scale_to_snr(audio, mix, snr_db)).astype(np.float32)

def add_events(audio, sr, rng, bank="cough", snr_db=10, n_events=2, **k):
    nb = BANKS.get(bank)
    if not nb or not nb.files:
        return audio
    out = audio.copy()
    for _ in range(n_events):
        ev = nb.event(rng)
        if len(ev) == 0:
            continue
        ev = ev[:len(audio)]
        pos = int(rng.integers(0, max(1, len(audio) - len(ev))))
        seg = out[pos:pos + len(ev)]
        ev = scale_to_snr(audio, ev[:len(seg)], snr_db)
        out[pos:pos + len(ev)] += ev
    return out.astype(np.float32)

def synth_hum(audio, sr, rng, snr_db=28, base=50.0, n_harm=4, **k):
    t = np.arange(len(audio)) / sr
    hum = sum(np.sin(2 * np.pi * base * h * t) / h for h in range(1, n_harm + 1))
    return (audio + scale_to_snr(audio, hum.astype(np.float32), snr_db)).astype(np.float32)

def synth_beeps(audio, sr, rng, snr_db=22, freq=1000.0, beep_ms=150, interval=5.0, **k):
    bn = int(sr * beep_ms / 1000.0)
    beep = (np.sin(2 * np.pi * freq * np.arange(bn) / sr) * np.hanning(bn)).astype(np.float32)
    tmpl = np.zeros(len(audio), np.float32); step = max(bn, int(sr * interval))
    for pos in range(0, len(audio) - bn, step):
        tmpl[pos:pos + bn] += beep
    return (audio + scale_to_snr(audio, tmpl, snr_db)).astype(np.float32)

def bandlimit(audio, sr, rng, low=120.0, high=6000.0, order=4, **k):
    high = min(high, sr / 2 - 1)
    sos = butter(order, [low, high], btype="band", fs=sr, output="sos")
    return sosfilt(sos, audio).astype(np.float32)

def clip_dist(audio, sr, rng, drive=0.2, **k):
    return np.clip(audio * (1.0 + drive * 6.0), -1.0, 1.0).astype(np.float32)

_CODECS = {"opus": ("opus", ["-c:a", "libopus"]),
           "mp3":  ("mp3",  ["-c:a", "libmp3lame"]),
           "aac":  ("m4a",  ["-c:a", "aac"])}
def codec_roundtrip(audio, sr, rng, codec="opus", bitrate="24k", **k):
    ext, enc = _CODECS.get(codec, _CODECS["opus"])
    try:
        with tempfile.TemporaryDirectory() as d:
            wi, ec, wo = (os.path.join(d, f) for f in ("in.wav", "e." + ext, "out.wav"))
            sf.write(wi, np.clip(audio, -1, 1).astype(np.float32), sr, subtype="PCM_16")
            subprocess.run(["ffmpeg", "-y", "-i", wi, *enc, "-b:a", bitrate, ec],
                           check=True, capture_output=True)
            subprocess.run(["ffmpeg", "-y", "-i", ec, "-ar", str(sr), "-ac", "1", wo],
                           check=True, capture_output=True)
            y, _ = sf.read(wo, dtype="float32")
        return _fit(np.asarray(y, np.float32), len(audio))
    except Exception as e:
        print(f"    [codec_roundtrip skipped: {e}]")
        return audio

EFFECTS = {
    "reverb_pyroom":   (reverb_pyroom,   "reverb"),
    "reverb_synth":    (reverb_synth,    "reverb"),
    "add_gaussian":    (add_gaussian,    "noise"),
    "add_noise":       (add_noise,       "noise"),
    "add_babble":      (add_babble,      "noise"),
    "add_events":      (add_events,      "noise"),
    "synth_hum":       (synth_hum,       "noise"),
    "synth_beeps":     (synth_beeps,     "noise"),
    "bandlimit":       (bandlimit,       "mic"),
    "clip_dist":       (clip_dist,       "mic"),
    "codec_roundtrip": (codec_roundtrip, "mic"),
}

# ── the two clinic profiles (EDIT HERE) ──────────────────────────
# Each line is (effect_name, params). reverb + mic are fixed channel effects.
# 'noise' effects ride the sweep: snr_db is shifted by (S - PROFILE_REF_SNR), so at
# S == PROFILE_REF_SNR the mix is exactly the params below (realistic Gaussian SNR == S).
REALISTIC_PIPELINE = [
    ("reverb_pyroom",   dict(rt60=0.45, room=(4.0, 5.0, 3.0))),     # small tiled exam room
    ("add_gaussian",    dict(snr_db=PROFILE_REF_SNR)),              # cheap-mic self-noise -> swept to S
    ("bandlimit",       dict(low=120.0, high=6000.0)),              # cheap-mic rolloff
    ("clip_dist",       dict(drive=0.15)),                          # mild overdrive
    ("codec_roundtrip", dict(codec="opus", bitrate="20k")),         # VoIP / cheap capture
]
WORST_PIPELINE = [
    ("reverb_pyroom",   dict(rt60=0.45, room=(4.0, 5.0, 3.0))),
    ("add_babble",      dict(snr_db=15, n_voices=4)),               # waiting-room chatter
    ("add_events",      dict(bank="cough", snr_db=8, n_events=2)),  # patient cough/breath
    ("synth_hum",       dict(snr_db=28, base=50.0, n_harm=4)),      # HVAC / mains hum
    ("synth_beeps",     dict(snr_db=22, freq=1000.0, interval=5.0)),# monitor beep
    ("add_gaussian",    dict(snr_db=20)),                           # cheap-mic floor
    ("bandlimit",       dict(low=120.0, high=6000.0)),
    ("clip_dist",       dict(drive=0.15)),
    ("codec_roundtrip", dict(codec="opus", bitrate="20k")),
]

def apply_profile(audio, sr, pipeline, snr_value, rng, ref_snr=None):
    """Apply a clinic pipeline; 'noise' effects shift by (snr_value - ref_snr)."""
    if ref_snr is None:
        ref_snr = PROFILE_REF_SNR
    x = np.asarray(audio, np.float32).copy()
    offset = snr_value - ref_snr
    for name, params in pipeline:
        fn, cat = EFFECTS[name]
        p = dict(params)
        if cat == "noise":
            p["snr_db"] = p.get("snr_db", ref_snr) + offset
        try:
            x = fn(x, sr, rng, **p)
        except Exception as e:
            print(f"    [warn] {name} failed: {e}")
    return x.astype(np.float32)

print("Synthetic noise engine ready |", len(EFFECTS), "effects | profiles: realistic, worst")


Dataset loaders — every loader returns the **same `clips` schema** (plus a `meta` dict carried through to the CSV), so all model cells below are unchanged.

In [ ]:
# ════════════════════════════════════════════════════════════════
#  DATASET LOADERS  →  build the `clips` list every model iterates over
# ════════════════════════════════════════════════════════════════
def _to_mono_f32(arr):
    arr = np.asarray(arr, dtype=np.float32)
    if arr.ndim > 1:
        arr = arr.mean(axis=1).astype(np.float32)   # [samples, channels] -> mono
    return arr

def _decode_audio(val):
    """Return (np.float32 mono array, sample_rate) from an HF audio value."""
    # new datasets torchcodec backend
    if val.__class__.__name__ == "AudioDecoder":
        s   = val.get_all_samples()
        arr = s.data.numpy() if hasattr(s.data, "numpy") else np.asarray(s.data)
        if arr.ndim > 1:
            arr = arr.mean(axis=0)                    # [channels, samples] -> mono
        return arr.astype(np.float32), int(s.sample_rate)
    if isinstance(val, dict):
        if val.get("array") is not None:
            return _to_mono_f32(val["array"]), int(val["sampling_rate"])
        if val.get("bytes"):
            a, sr = sf.read(io.BytesIO(val["bytes"]), dtype="float32")
            return _to_mono_f32(a), int(sr)
        if val.get("path"):
            a, sr = sf.read(val["path"], dtype="float32")
            return _to_mono_f32(a), int(sr)
    if isinstance(val, str):
        a, sr = sf.read(val, dtype="float32")
        return _to_mono_f32(a), int(sr)
    raise ValueError(f"Unrecognized audio value type: {type(val)}")

def _load_fleurs_ds():
    if os.path.exists(FLEURS_DISK_PATH):
        print(f"Loading FLEURS fa_ir from attached dataset: {FLEURS_DISK_PATH}")
        return load_from_disk(FLEURS_DISK_PATH)
    print("FLEURS not attached — downloading google/fleurs fa_ir test from HuggingFace ...")
    ds = load_dataset("google/fleurs", "fa_ir", split="test")
    ds.save_to_disk("/kaggle/working/fleurs_fa_ir_test")
    return ds

def build_clips_fleurs():
    ds = _load_fleurs_ds().cast_column("audio", Audio(decode=False))
    clips = []
    for i, row in enumerate(ds):
        arr, sr = _decode_audio(row["audio"])
        ref = (row.get("transcription") or row.get("raw_transcription") or "").strip()
        clips.append({"clip_id": i, "audio_array": arr, "sample_rate": sr,
                      "duration_sec": len(arr) / sr, "reference": ref, "meta": {}})
    return clips

def build_clips_psrb():
    # PSRB ships transcripts in Labels.csv (+ wavs in Files/). NOTE: that CSV's
    # header is MISALIGNED with its data — the transcript actually lives under the
    # column labeled 'audio_duration', and 'text' sits over number_of_speakers.
    # So we detect columns by CONTENT, not by header name.
    from huggingface_hub import snapshot_download
    local  = snapshot_download(repo_id=PSRB_HF_ID, repo_type="dataset")
    labels = pd.read_csv(os.path.join(local, "Labels.csv"))
    print(f"PSRB Labels.csv: {len(labels)} rows. Columns: {list(labels.columns)}")

    fa     = re.compile(r'[\u0600-\u06FF]')          # Arabic/Persian block
    sample = labels.head(50).astype(str)
    path_col = next((c for c in labels.columns
                     if sample[c].str.contains(r'\.wav', case=False).mean() > 0.5),
                    labels.columns[0])
    text_col = max((c for c in labels.columns if c != path_col),
                   key=lambda c: sample[c].map(lambda s: len(fa.findall(s))).mean())
    print(f"Detected  path_col='{path_col}'  text_col='{text_col}'  (header names ignored)")

    if "acoustic_environment" in labels.columns:
        print("acoustic_environment distribution:",
              dict(Counter(labels["acoustic_environment"].astype(str))))
    if PSRB_DROP_CLEAN and "acoustic_environment" in labels.columns:
        labels = labels[labels["acoustic_environment"].astype(str).str.strip().str.lower() != "clean"]
        print(f"After dropping 'clean': {len(labels)} rows")
    if PSRB_SPONTANEOUS_ONLY and "spontaneous" in labels.columns:
        labels = labels[labels["spontaneous"].astype(str).isin(["1","True","true"])]
        print(f"After spontaneous-only: {len(labels)} rows")

    clips, missing = [], 0
    for i, row in labels.reset_index(drop=True).iterrows():
        rel  = str(row[path_col])
        cand = os.path.join(local, rel)
        if not os.path.exists(cand):
            cand = os.path.join(local, "Files", os.path.basename(rel))
        if not os.path.exists(cand):
            missing += 1; continue
        arr, sr = sf.read(cand, dtype="float32")
        arr = _to_mono_f32(arr)
        ref = str(row[text_col]).strip()
        meta = {"acoustic_environment": row.get("acoustic_environment"),
                "spontaneous": row.get("spontaneous")}
        clips.append({"clip_id": int(i), "audio_array": arr, "sample_rate": int(sr),
                      "duration_sec": len(arr) / sr, "reference": ref, "meta": meta})
    if missing:
        print(f"WARNING: {missing} audio files not found and skipped.")
    return clips

def build_clips_common_voice():
    # hezarai mirror is public; mozilla candidates need a token
    last_err = None
    ds = None
    for hf_id in CV_HF_CANDIDATES:
        is_mozilla = hf_id.startswith("mozilla-foundation")
        if is_mozilla and not CV_HF_TOKEN:
            print(f"  Skipping {hf_id} (no HF_TOKEN set)")
            continue
        config = "fa" if is_mozilla else CV_CONFIG
        try:
            print(f"Trying {hf_id} (config={config}, split={CV_SPLIT}) ...")
            ds = load_dataset(hf_id, config, split=CV_SPLIT,
                              token=CV_HF_TOKEN if is_mozilla else None)
            print(f"Loaded: {hf_id}  ({len(ds)} rows)")
            break
        except Exception as e:
            msg = str(e)
            if "doesn't contain any data files" in msg:
                print(f"  ✗ {hf_id}: token rejected or terms not accepted")
            else:
                print(f"  ✗ {hf_id}: {msg}")
            last_err = e

    if ds is None:
        raise RuntimeError(
            "All Common Voice candidates failed.\n"
            "  - hezarai/common-voice-13-fa should be public — check internet access.\n"
            "  - For mozilla candidates: set HF_TOKEN and accept terms on the HF dataset page.\n"
            f"Last error: {last_err}"
        )

    audio_col = "audio" if "audio" in ds.column_names else ds.column_names[0]
    ref_col   = next((c for c in ("sentence", "transcription", "text") if c in ds.column_names), None)

    if CV_MAX_CLIPS and CV_MAX_CLIPS < len(ds):
        ds = ds.shuffle(seed=42).select(range(CV_MAX_CLIPS))
        print(f"Randomly sampled {CV_MAX_CLIPS} clips from {len(ds)} (seed=42)")

    ds = ds.cast_column(audio_col, Audio(decode=False))
    clips = []
    for i, row in enumerate(ds):
        arr, sr = _decode_audio(row[audio_col])
        ref = (row.get(ref_col) or "").strip() if ref_col else ""
        clips.append({"clip_id": i, "audio_array": arr, "sample_rate": sr,
                      "duration_sec": len(arr) / sr, "reference": ref, "meta": {}})
    return clips

def _add_noise_at_snr(clean, snr_db, rng):
    """Additive white Gaussian noise scaled to the requested SNR.
    PLACEHOLDER for MUSAN/DEMAND/RIR — swap the noise source here in Round 3."""
    p_sig   = float(np.mean(clean.astype(np.float64) ** 2)) + 1e-12
    p_noise = p_sig / (10.0 ** (snr_db / 10.0))
    noise   = rng.normal(0.0, np.sqrt(p_noise), size=clean.shape).astype(np.float32)
    return (clean + noise).astype(np.float32)

def build_clips_synthetic():
    print("Synthetic round — additive Gaussian noise PLACEHOLDER "
          "(replace with MUSAN/DEMAND/RIR once chosen).")
    base = build_clips_fleurs()
    if SYNTH_MAX_CLIPS:
        base = base[:SYNTH_MAX_CLIPS]
    rng = np.random.default_rng(1234)
    clips = []
    for c in base:
        for snr in SYNTH_SNR_DB:
            noisy = _add_noise_at_snr(c["audio_array"], snr, rng)
            clips.append({"clip_id": c["clip_id"],
                          "audio_array": noisy, "sample_rate": c["sample_rate"],
                          "duration_sec": c["duration_sec"], "reference": c["reference"],
                          "meta": {"snr_db": snr, "noise_type": SYNTH_NOISE}})
    return clips

def _ensure_synth_sr(arr, sr):
    arr = np.asarray(arr, np.float32)
    if sr == SYNTH_SR or len(arr) == 0:
        return arr, (SYNTH_SR if len(arr) else sr)
    g = gcd(int(sr), SYNTH_SR)
    return resample_poly(arr, SYNTH_SR // g, sr // g).astype(np.float32), SYNTH_SR

def _build_synthetic_profile(pipeline, profile_name):
    base = build_clips_fleurs()
    if SYNTH_MAX_CLIPS:
        base = base[:SYNTH_MAX_CLIPS]
    rng = np.random.default_rng(1234)
    clips = []
    for c in base:
        arr, sr = _ensure_synth_sr(c["audio_array"], c["sample_rate"])
        for snr in SYNTH_SNR_DB:
            noisy = apply_profile(arr, sr, pipeline, snr, rng)
            clips.append({"clip_id": c["clip_id"], "audio_array": noisy,
                          "sample_rate": sr, "duration_sec": len(noisy) / sr,
                          "reference": c["reference"],
                          "meta": {"snr_db": snr, "noise_type": profile_name}})
    print(f"  built {len(clips)} clips = {len(base)} FLEURS x {len(SYNTH_SNR_DB)} SNRs [{profile_name}]")
    return clips

def build_clips_synthetic_realistic():
    print("Synthetic 3a - clinic-REALISTIC (reverb + cheap-mic + Gaussian), SNR sweep.")
    return _build_synthetic_profile(REALISTIC_PIPELINE, "clinic_realistic")

def build_clips_synthetic_worst():
    print("Synthetic 3b - clinic-WORST-CASE (reverb + babble + cough + hum + beeps + Gaussian + mic), SNR sweep.")
    if not BANKS["babble"].files or not BANKS["cough"].files:
        print("  [WARN] MUSAN/ESC-50 banks empty -> babble/cough skipped. "
              "Attach the noise datasets for the full worst-case profile.")
    return _build_synthetic_profile(WORST_PIPELINE, "clinic_worst")

_BUILDERS = {
    "fleurs_clean":         build_clips_fleurs,
    "psrb":                 build_clips_psrb,
    "common_voice_fa":      build_clips_common_voice,
    "synthetic":            build_clips_synthetic,
    "synthetic_realistic":  build_clips_synthetic_realistic,
    "synthetic_worst":      build_clips_synthetic_worst,
}
if DATASET not in _BUILDERS:
    raise ValueError(f"Unknown DATASET '{DATASET}'. Choose from {list(_BUILDERS)}")

clips = _BUILDERS[DATASET]()
total_min = sum(c["duration_sec"] for c in clips) / 60
print(f"\nDATASET={DATASET}: {len(clips)} clips — {total_min:.1f} min audio")
print(f"Sample rate (first clip): {clips[0]['sample_rate']} Hz")
print(f"Example reference: {clips[0]['reference'][:80]}")

Persian text normalizer, helper scorers, and the `evaluate_model` runner. Same proven core as the clean benchmark, extended for the noisy round: bootstrap WER CI, `is_hallucination` flag, ΔWER vs the clean baseline, and `meta` passthrough.

In [17]:
# ════════════════════════════════════════════════════════════════
#  SHARED UTILITIES
# ════════════════════════════════════════════════════════════════
_hazm_norm   = hazm.Normalizer()
_chrf_metric = CHRF()

_PERSIAN_RANGES = [('؀', 'ۿ'), ('ﭐ', '﷿'), ('ﹰ', '﻿')]

def _is_persian_char(c):
    return any(lo <= c <= hi for lo, hi in _PERSIAN_RANGES)

def normalize_persian(text: str) -> str:
    """Hazm-normalize Persian text and strip punctuation (refs AND hyps)."""
    text = _hazm_norm.normalize(text)
    text = re.sub(r"[^\w\s]", "", text, flags=re.UNICODE)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def _script_contamination(text: str) -> float:
    """Fraction of alphabetic characters that are NOT Persian/Arabic script."""
    alpha = [c for c in text if c.isalpha()]
    if not alpha:
        return 0.0
    non_persian = sum(1 for c in alpha if not _is_persian_char(c))
    return non_persian / len(alpha)

def _repetition_score(text: str, n: int = 4) -> float:
    """Fraction of 4-grams that are repeated (Whisper hallucination signature)."""
    words = text.split()
    if len(words) < n + 1:
        return 0.0
    ngrams = [tuple(words[i:i+n]) for i in range(len(words) - n + 1)]
    counts = Counter(ngrams)
    repeated = sum(v - 1 for v in counts.values() if v > 1)
    return repeated / len(ngrams)

def _punct_f1(ref: str, hyp: str) -> float:
    """F1 for Persian punctuation marks."""
    PUNCTS = set('.,،؟!؛:')
    r = [c for c in ref if c in PUNCTS]
    h = [c for c in hyp if c in PUNCTS]
    if not r and not h:
        return 1.0
    if not r or not h:
        return 0.0
    rc, hc = Counter(r), Counter(h)
    match = sum((rc & hc).values())
    prec  = match / len(h)
    rec   = match / len(r)
    return 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0

# ── noisy-round additions ────────────────────────────────────────
def _is_hallucination(rep, ratio, contam, is_empty):
    """One composite flag: confident, fluent output that was not said."""
    if is_empty:
        return 0
    return int(rep > 0.2 or ratio > 2.0 or contam > 0.5)

def _bootstrap_wer_ci(errs, refs, B=1000, alpha=0.05, seed=0):
    """Bootstrap CI for micro-averaged corpus WER, resampling over clips."""
    errs = np.asarray(errs, dtype=float)
    refs = np.asarray(refs, dtype=float)
    n = len(errs)
    if n == 0 or refs.sum() == 0:
        return (float("nan"), float("nan"))
    rng  = np.random.default_rng(seed)
    vals = np.empty(B, dtype=float)
    for b in range(B):
        idx   = rng.integers(0, n, n)
        denom = refs[idx].sum()
        vals[b] = errs[idx].sum() / denom if denom > 0 else np.nan
    lo = float(np.nanpercentile(vals, 100 * alpha / 2))
    hi = float(np.nanpercentile(vals, 100 * (1 - alpha / 2)))
    return (round(lo, 4), round(hi, 4))

def _load_clean_baseline(safe):
    """clip_id -> clean WER, from a fleurs_clean per-model CSV (for ΔWER)."""
    path = os.path.join(CLEAN_BASELINE_DIR, f"results__{safe}.csv")
    if not os.path.exists(path):
        return {}
    b = pd.read_csv(path)
    b = b[b["clip_id"].astype(str) != "SUMMARY"]
    out = {}
    for _, r in b.iterrows():
        try:
            out[int(r["clip_id"])] = float(r["wer"])
        except Exception:
            pass
    return out

def free_gpu():
    """Release GPU memory after each model."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def evaluate_model(transcribe_fn, clips, model_name):
    """Run transcribe_fn on every clip, compute all metrics, save CSV to OUT_DIR.

    transcribe_fn may return `hyp_text` OR `(hyp_text, meta_dict)` where meta_dict
    can carry e.g. {"no_speech_prob": float}. All model cells below return plain
    text (unchanged from the proven clean benchmark)."""
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

    safe     = model_name.replace("/", "__")
    baseline = _load_clean_baseline(safe) if PAIRED_WITH_CLEAN else {}

    print(f"\nEvaluating: {model_name}  ({len(clips)} clips)  [{DATASET}]")
    records, err_list, refw_list = [], [], []

    for i, clip in enumerate(clips):
        t0 = time.perf_counter()
        try:
            res = transcribe_fn(clip["audio_array"], clip["sample_rate"])
        except Exception as exc:
            res = ""
            print(f"  [WARN] clip {clip['clip_id']} failed: {exc}")
        elapsed = time.perf_counter() - t0

        if isinstance(res, tuple):
            hyp, meta_extra = res[0], (res[1] or {})
        else:
            hyp, meta_extra = res, {}
        no_speech_prob = meta_extra.get("no_speech_prob", float("nan"))

        ref   = clip["reference"]
        ref_n = normalize_persian(ref)
        hyp_n = normalize_persian(hyp)

        try:
            clip_wer = wer(ref_n, hyp_n) if ref_n else float("nan")
            clip_cer = cer(ref_n, hyp_n) if ref_n else float("nan")
        except Exception:
            clip_wer = clip_cer = float("nan")

        # error counts for the bootstrap CI (micro-averaged WER)
        if ref_n:
            try:
                pw_c  = process_words([ref_n], [hyp_n])
                n_err = pw_c.substitutions + pw_c.insertions + pw_c.deletions
                n_ref = pw_c.hits + pw_c.substitutions + pw_c.deletions
            except Exception:
                n_err = n_ref = 0
            if n_ref > 0:
                err_list.append(n_err); refw_list.append(n_ref)

        try:
            clip_chrf = _chrf_metric.sentence_score(hyp_n, [ref_n]).score if ref_n else float("nan")
        except Exception:
            clip_chrf = float("nan")

        ref_words  = len(ref_n.split()) if ref_n else 0
        hyp_words  = len(hyp_n.split())
        hall_ratio = round(hyp_words / max(ref_words, 1), 4)
        is_empty   = int(hyp.strip() == "")
        contam     = round(_script_contamination(hyp), 4)
        rep        = round(_repetition_score(hyp), 4)

        # ΔWER vs clean baseline (paired rounds only)
        if baseline and clip_wer == clip_wer:            # clip_wer not NaN
            base_wer  = baseline.get(int(clip["clip_id"]))
            delta_wer = round(clip_wer - base_wer, 4) if base_wer is not None else float("nan")
        else:
            delta_wer = float("nan")

        m = clip.get("meta", {})
        records.append({
            "clip_id"             : clip["clip_id"],
            "duration_sec"        : round(clip["duration_sec"], 3),
            "reference"           : ref,
            "hypothesis"          : hyp,
            "wer"                 : round(clip_wer, 4),
            "cer"                 : round(clip_cer, 4),
            "chrf"                : round(clip_chrf, 4) if not np.isnan(clip_chrf) else float("nan"),
            "hallucination_ratio" : hall_ratio,
            "is_empty"            : is_empty,
            "is_hallucination"    : _is_hallucination(rep, hall_ratio, contam, is_empty),
            "script_contamination": contam,
            "repetition_score"    : rep,
            "punct_f1"            : round(_punct_f1(ref, hyp), 4),
            "delta_wer"           : delta_wer,
            "no_speech_prob"      : no_speech_prob,
            "snr_db"              : m.get("snr_db"),
            "noise_type"          : m.get("noise_type"),
            "acoustic_environment": m.get("acoustic_environment"),
            "spontaneous"         : m.get("spontaneous"),
            "inference_time_sec"  : round(elapsed, 4),
            "rtf"                 : round(elapsed / clip["duration_sec"], 4),
        })

        if (i + 1) % 100 == 0:
            avg_wer = np.nanmean([r["wer"] for r in records])
            print(f"  [{i + 1}/{len(clips)}]  running avg WER = {avg_wer:.4f}")

    df    = pd.DataFrame(records)
    valid = df[df["reference"].astype(str).str.strip() != ""]

    # ── CORPUS METRICS ───────────────────────────────────────────
    ref_list = [normalize_persian(r) for r in valid["reference"]]
    hyp_list = [normalize_persian(h) for h in valid["hypothesis"].fillna("")]
    ref_corp = " ".join(ref_list)
    hyp_corp = " ".join(hyp_list)

    corp_wer = round(wer(ref_corp, hyp_corp), 4) if ref_corp else float("nan")
    corp_cer = round(cer(ref_corp, hyp_corp), 4) if ref_corp else float("nan")
    try:
        corp_chrf = round(_chrf_metric.corpus_score(hyp_list, [ref_list]).score, 4)
    except Exception:
        corp_chrf = float("nan")

    wer_ci_low, wer_ci_high = _bootstrap_wer_ci(
        err_list, refw_list, B=WER_CI_BOOTSTRAP, alpha=WER_CI_ALPHA)

    # ── ERROR BREAKDOWN ──────────────────────────────────────────
    try:
        pw              = process_words(ref_list, hyp_list)
        total_ref_words = pw.hits + pw.substitutions + pw.deletions
        sub_rate = round(pw.substitutions / total_ref_words, 4) if total_ref_words else float("nan")
        ins_rate = round(pw.insertions    / total_ref_words, 4) if total_ref_words else float("nan")
        del_rate = round(pw.deletions     / total_ref_words, 4) if total_ref_words else float("nan")
    except Exception:
        sub_rate = ins_rate = del_rate = float("nan")

    # ── WER DISTRIBUTION ─────────────────────────────────────────
    wer_vals         = df["wer"].dropna().values
    wer_p50          = round(float(np.percentile(wer_vals, 50)), 4) if len(wer_vals) else float("nan")
    wer_p90          = round(float(np.percentile(wer_vals, 90)), 4) if len(wer_vals) else float("nan")
    wer_p95          = round(float(np.percentile(wer_vals, 95)), 4) if len(wer_vals) else float("nan")
    pct_perfect      = round(float(np.mean(wer_vals == 0)),      4) if len(wer_vals) else float("nan")
    pct_catastrophic = round(float(np.mean(wer_vals > 0.5)),     4) if len(wer_vals) else float("nan")
    ser              = round(float(np.mean(wer_vals > 0)),       4) if len(wer_vals) else float("nan")
    pct_hallucinated = round(float(valid["is_hallucination"].mean()), 4) if len(valid) else float("nan")
    avg_delta_wer    = round(float(np.nanmean(df["delta_wer"].values)), 4) if df["delta_wer"].notna().any() else float("nan")

    # ── DEPLOYMENT METRICS ───────────────────────────────────────
    empty_rate = round(df["is_empty"].mean(), 4)
    avg_inf    = round(df["inference_time_sec"].mean(), 4)
    avg_rtf    = round(df["rtf"].mean(), 4)
    throughput = round(1.0 / avg_rtf, 4) if avg_rtf > 0 else float("nan")
    peak_vram  = round(torch.cuda.max_memory_allocated() / 1e9, 3) if torch.cuda.is_available() else 0.0

    # ── WER BY DURATION BUCKET ──────────────────────────────────
    def _bucket_wer(lo, hi):
        mask = (df["duration_sec"] >= lo) & (df["duration_sec"] < hi)
        vals = df.loc[mask, "wer"].dropna()
        return round(float(np.nanmean(vals)), 4) if len(vals) else float("nan")
    wer_lt5s   = _bucket_wer(0,  5)
    wer_5to15s = _bucket_wer(5,  15)
    wer_gt15s  = _bucket_wer(15, 9999)

    # ── SUMMARY ROW ─────────────────────────────────────────────
    summary = {
        "clip_id": "SUMMARY", "duration_sec": round(df["duration_sec"].sum(), 2),
        "reference": "", "hypothesis": "",
        "wer": corp_wer, "cer": corp_cer, "chrf": corp_chrf,
        "hallucination_ratio": round(df["hallucination_ratio"].mean(), 4),
        "is_empty": empty_rate, "is_hallucination": pct_hallucinated,
        "script_contamination": round(df["script_contamination"].mean(), 4),
        "repetition_score": round(df["repetition_score"].mean(), 4),
        "punct_f1": round(df["punct_f1"].mean(), 4),
        "delta_wer": avg_delta_wer,
        "no_speech_prob": round(float(np.nanmean(df["no_speech_prob"].values)), 4) if df["no_speech_prob"].notna().any() else float("nan"),
        "snr_db": "", "noise_type": "", "acoustic_environment": "", "spontaneous": "",
        "inference_time_sec": avg_inf, "rtf": avg_rtf,
        "ser": ser, "wer_p50": wer_p50, "wer_p90": wer_p90, "wer_p95": wer_p95,
        "pct_perfect": pct_perfect, "pct_catastrophic": pct_catastrophic,
        "wer_ci_low": wer_ci_low, "wer_ci_high": wer_ci_high,
        "sub_rate": sub_rate, "ins_rate": ins_rate, "del_rate": del_rate,
        "throughput_audio_hrs_per_hr": throughput, "peak_vram_gb": peak_vram,
        "wer_lt5s": wer_lt5s, "wer_5to15s": wer_5to15s, "wer_gt15s": wer_gt15s,
    }
    df = pd.concat([df, pd.DataFrame([summary])], ignore_index=True)

    csv_path = os.path.join(OUT_DIR, f"results__{safe}.csv")
    df.to_csv(csv_path, index=False)

    sep = "=" * 66
    print(f"\n{sep}")
    print(f"  {model_name}   [{DATASET}]")
    print(f"  Corpus WER         : {corp_wer:.4f}   95% CI [{wer_ci_low:.4f}, {wer_ci_high:.4f}]")
    print(f"  WER P50/P90/P95    : {wer_p50:.3f} / {wer_p90:.3f} / {wer_p95:.3f}")
    print(f"  Corpus CER / chrF  : {corp_cer:.4f} / {corp_chrf:.2f}")
    print(f"  SER / perfect / catastrophic : {ser:.3f} / {pct_perfect:.1%} / {pct_catastrophic:.1%}")
    print(f"  Hallucinated / empty : {pct_hallucinated:.1%} / {empty_rate:.1%}")
    print(f"  Sub/Ins/Del        : {sub_rate:.4f} / {ins_rate:.4f} / {del_rate:.4f}")
    if not np.isnan(avg_delta_wer):
        print(f"  Mean ΔWER vs clean : {avg_delta_wer:+.4f}")
    print(f"  RTF {avg_rtf:.4f}  ({throughput:.2f}x real-time)   peak VRAM {peak_vram:.2f} GB")
    print(f"  Saved -> {csv_path}")
    print(sep)
    return df

---
## 1 · Whisper (stock OpenAI)

### 1.1 — whisper-large-v3

In [18]:
MODEL_ID    = "openai/whisper-large-v3"
_dtype      = torch.float16 if device == "cuda" else torch.float32
print(f"Loading {MODEL_ID} ...")
w_proc  = WhisperProcessor.from_pretrained(MODEL_ID)
w_model = (
    WhisperForConditionalGeneration
    .from_pretrained(MODEL_ID, torch_dtype=_dtype)
    .to(device).eval()
)
print("Ready.")

Loading openai/whisper-large-v3 ...


Loading weights:   0%|          | 0/1259 [00:00<?, ?it/s]

Ready.


Transcribe with `whisper-large-v3`, compute metrics, save CSV.

In [20]:
def _t(audio, sr):
    if sr != 16000:
        audio = torchaudio.functional.resample(
            torch.from_numpy(audio).unsqueeze(0), sr, 16000
        ).squeeze(0).numpy()
    feat = w_proc(audio, sampling_rate=16000, return_tensors="pt").input_features.to(device)
    if device == "cuda":
        feat = feat.half()
    with torch.no_grad():
        ids = w_model.generate(feat, language="fa", task="transcribe")
    return w_proc.batch_decode(ids, skip_special_tokens=True)[0]

df1 = evaluate_model(_t, clips, "openai/whisper-large-v3")
del w_model, w_proc; free_gpu()


Evaluating: openai/whisper-large-v3  (183 clips)  [psrb]
  [100/183]  running avg WER = 0.3999

  openai/whisper-large-v3   [psrb]
  Corpus WER         : 0.3808   95% CI [0.3445, 0.4290]
  WER P50/P90/P95    : 0.323 / 0.763 / 0.994
  Corpus CER / chrF  : 0.1756 / 71.91
  SER / perfect / catastrophic : 0.962 / 3.8% / 26.8%
  Hallucinated / empty : 0.0% / 0.0%
  Sub/Ins/Del        : 0.2431 / 0.0348 / 0.1061
  RTF 0.2695  (3.71x real-time)   peak VRAM 5.30 GB
  Saved -> /kaggle/working/psrb/results__openai__whisper-large-v3.csv


### 1.2 — whisper-large-v3-turbo

In [21]:
MODEL_ID    = "openai/whisper-large-v3-turbo"
_dtype      = torch.float16 if device == "cuda" else torch.float32
print(f"Loading {MODEL_ID} ...")
w_proc  = WhisperProcessor.from_pretrained(MODEL_ID)
w_model = (
    WhisperForConditionalGeneration
    .from_pretrained(MODEL_ID, torch_dtype=_dtype)
    .to(device).eval()
)
print("Ready.")

Loading openai/whisper-large-v3-turbo ...


Loading weights:   0%|          | 0/587 [00:00<?, ?it/s]

Ready.


Transcribe with `whisper-large-v3-turbo`, compute metrics, save CSV.

In [22]:
def _t(audio, sr):
    if sr != 16000:
        audio = torchaudio.functional.resample(
            torch.from_numpy(audio).unsqueeze(0), sr, 16000
        ).squeeze(0).numpy()
    feat = w_proc(audio, sampling_rate=16000, return_tensors="pt").input_features.to(device)
    if device == "cuda":
        feat = feat.half()
    with torch.no_grad():
        ids = w_model.generate(feat, language="fa", task="transcribe")
    return w_proc.batch_decode(ids, skip_special_tokens=True)[0]

df2 = evaluate_model(_t, clips, "openai/whisper-large-v3-turbo")
del w_model, w_proc; free_gpu()


Evaluating: openai/whisper-large-v3-turbo  (183 clips)  [psrb]
  [100/183]  running avg WER = 0.8927

  openai/whisper-large-v3-turbo   [psrb]
  Corpus WER         : 0.5486   95% CI [0.4213, 0.7279]
  WER P50/P90/P95    : 0.375 / 0.852 / 1.000
  Corpus CER / chrF  : 0.3002 / 68.05
  SER / perfect / catastrophic : 0.973 / 2.7% / 30.0%
  Hallucinated / empty : 2.2% / 0.0%
  Sub/Ins/Del        : 0.2807 / 0.1696 / 0.0998
  RTF 0.0846  (11.82x real-time)   peak VRAM 5.02 GB
  Saved -> /kaggle/working/psrb/results__openai__whisper-large-v3-turbo.csv


---
## 2 · Whisper (fine-tuned Persian)

### 2.1 — persian-whisper-large-v3 (Halakoo)

In [23]:
MODEL_ID    = "MohammadReza-Halakoo/persian-whisper-large-v3-10-percent-17-0-one-epoch"
_dtype      = torch.float16 if device == "cuda" else torch.float32
print(f"Loading {MODEL_ID} ...")
w_proc  = WhisperProcessor.from_pretrained(MODEL_ID)
w_model = (
    WhisperForConditionalGeneration
    .from_pretrained(MODEL_ID, torch_dtype=_dtype)
    .to(device).eval()
)
print("Ready.")

Loading MohammadReza-Halakoo/persian-whisper-large-v3-10-percent-17-0-one-epoch ...


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1259 [00:00<?, ?it/s]

Ready.


Transcribe with the Halakoo fine-tune, compute metrics, save CSV.

In [24]:
def _t(audio, sr):
    if sr != 16000:
        audio = torchaudio.functional.resample(
            torch.from_numpy(audio).unsqueeze(0), sr, 16000
        ).squeeze(0).numpy()
    feat = w_proc(audio, sampling_rate=16000, return_tensors="pt").input_features.to(device)
    if device == "cuda":
        feat = feat.half()
    with torch.no_grad():
        ids = w_model.generate(feat)
    return w_proc.batch_decode(ids, skip_special_tokens=True)[0]

df3 = evaluate_model(_t, clips, "MohammadReza-Halakoo/persian-whisper-large-v3-10-percent-17-0-one-epoch")
del w_model, w_proc; free_gpu()


Evaluating: MohammadReza-Halakoo/persian-whisper-large-v3-10-percent-17-0-one-epoch  (183 clips)  [psrb]
  [100/183]  running avg WER = 0.3877

  MohammadReza-Halakoo/persian-whisper-large-v3-10-percent-17-0-one-epoch   [psrb]
  Corpus WER         : 0.3893   95% CI [0.3496, 0.4387]
  WER P50/P90/P95    : 0.333 / 0.755 / 0.856
  Corpus CER / chrF  : 0.1689 / 71.73
  SER / perfect / catastrophic : 0.945 / 5.5% / 25.7%
  Hallucinated / empty : 0.0% / 0.0%
  Sub/Ins/Del        : 0.2660 / 0.0311 / 0.0938
  RTF 0.2791  (3.58x real-time)   peak VRAM 6.78 GB
  Saved -> /kaggle/working/psrb/results__MohammadReza-Halakoo__persian-whisper-large-v3-10-percent-17-0-one-epoch.csv


### 2.2 — whisper-large-fa-v1 (vhdm)

In [25]:
MODEL_ID    = "vhdm/whisper-large-fa-v1"
_dtype      = torch.float16 if device == "cuda" else torch.float32
print(f"Loading {MODEL_ID} ...")
w_proc  = WhisperProcessor.from_pretrained(MODEL_ID)
w_model = (
    WhisperForConditionalGeneration
    .from_pretrained(MODEL_ID, torch_dtype=_dtype)
    .to(device).eval()
)
w_model.generation_config.forced_decoder_ids = None
print("Ready.")

Loading vhdm/whisper-large-fa-v1 ...


Loading weights:   0%|          | 0/587 [00:00<?, ?it/s]

Ready.


Transcribe with `whisper-large-fa-v1`, compute metrics, save CSV.

In [26]:
def _t(audio, sr):
    if sr != 16000:
        audio = torchaudio.functional.resample(
            torch.from_numpy(audio).unsqueeze(0), sr, 16000
        ).squeeze(0).numpy()
    feat = w_proc(audio, sampling_rate=16000, return_tensors="pt").input_features.to(device)
    if device == "cuda":
        feat = feat.half()
    with torch.no_grad():
        ids = w_model.generate(feat)
    return w_proc.batch_decode(ids, skip_special_tokens=True)[0]

df4 = evaluate_model(_t, clips, "vhdm/whisper-large-fa-v1")
del w_model, w_proc; free_gpu()


Evaluating: vhdm/whisper-large-fa-v1  (183 clips)  [psrb]
  [100/183]  running avg WER = 0.4077

  vhdm/whisper-large-fa-v1   [psrb]
  Corpus WER         : 0.5647   95% CI [0.4221, 0.7464]
  WER P50/P90/P95    : 0.333 / 0.785 / 0.995
  Corpus CER / chrF  : 0.2720 / 67.71
  SER / perfect / catastrophic : 0.902 / 9.8% / 31.7%
  Hallucinated / empty : 3.3% / 0.0%
  Sub/Ins/Del        : 0.3253 / 0.1862 / 0.0603
  RTF 0.0779  (12.84x real-time)   peak VRAM 5.02 GB
  Saved -> /kaggle/working/psrb/results__vhdm__whisper-large-fa-v1.csv


### 2.3 — whisper-persian-v4 (nezamisafa)

In [27]:
MODEL_ID    = "nezamisafa/whisper-persian-v4"
_dtype      = torch.float16 if device == "cuda" else torch.float32
print(f"Loading {MODEL_ID} ...")
w_proc  = WhisperProcessor.from_pretrained(MODEL_ID)
w_model = (
    WhisperForConditionalGeneration
    .from_pretrained(MODEL_ID, torch_dtype=_dtype)
    .to(device).eval()
)
w_model.generation_config.forced_decoder_ids = None
print("Ready.")

Loading nezamisafa/whisper-persian-v4 ...


preprocessor_config.json:   0%|          | 0.00/357 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1259 [00:00<?, ?it/s]

generation_config.json: 0.00B [00:00, ?B/s]

Ready.


Transcribe with `whisper-persian-v4`, compute metrics, save CSV.

In [28]:
def _t(audio, sr):
    if sr != 16000:
        audio = torchaudio.functional.resample(
            torch.from_numpy(audio).unsqueeze(0), sr, 16000
        ).squeeze(0).numpy()
    feat = w_proc(audio, sampling_rate=16000, return_tensors="pt").input_features.to(device)
    if device == "cuda":
        feat = feat.half()
    with torch.no_grad():
        ids = w_model.generate(feat)
    return w_proc.batch_decode(ids, skip_special_tokens=True)[0]

df5 = evaluate_model(_t, clips, "nezamisafa/whisper-persian-v4")
del w_model, w_proc; free_gpu()


Evaluating: nezamisafa/whisper-persian-v4  (183 clips)  [psrb]
  [100/183]  running avg WER = 0.3782

  nezamisafa/whisper-persian-v4   [psrb]
  Corpus WER         : 0.3882   95% CI [0.3460, 0.4382]
  WER P50/P90/P95    : 0.364 / 0.730 / 0.799
  Corpus CER / chrF  : 0.1628 / 72.10
  SER / perfect / catastrophic : 0.896 / 10.4% / 29.5%
  Hallucinated / empty : 0.0% / 0.0%
  Sub/Ins/Del        : 0.2834 / 0.0295 / 0.0772
  RTF 0.2933  (3.41x real-time)   peak VRAM 6.74 GB
  Saved -> /kaggle/working/psrb/results__nezamisafa__whisper-persian-v4.csv


---
## 3 · Wav2Vec2 XLSR (CTC)

### 3.1 — wav2vec2-large-xlsr-53-persian

In [29]:
MODEL_ID    = "jonatasgrosman/wav2vec2-large-xlsr-53-persian"
print(f"Loading {MODEL_ID} ...")
w2v_proc  = Wav2Vec2Processor.from_pretrained(MODEL_ID)
w2v_model = (
    Wav2Vec2ForCTC.from_pretrained(MODEL_ID)
    .to(device).eval()
)
print("Ready.")

Loading jonatasgrosman/wav2vec2-large-xlsr-53-persian ...


preprocessor_config.json:   0%|          | 0.00/158 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

vocab.json:   0%|          | 0.00/656 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.26G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.26G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/424 [00:00<?, ?it/s]

Ready.


Transcribe with Wav2Vec2, compute metrics, save CSV.

In [30]:
def _t(audio, sr):
    if sr != 16000:
        audio = torchaudio.functional.resample(
            torch.from_numpy(audio).unsqueeze(0), sr, 16000
        ).squeeze(0).numpy()
    inputs = w2v_proc(audio, sampling_rate=16000, return_tensors="pt", padding=True)
    with torch.no_grad():
        logits = w2v_model(inputs.input_values.to(device)).logits
    pred_ids = torch.argmax(logits, dim=-1)
    return w2v_proc.batch_decode(pred_ids)[0]

df6 = evaluate_model(_t, clips, "jonatasgrosman/wav2vec2-large-xlsr-53-persian")
del w2v_model, w2v_proc; free_gpu()


Evaluating: jonatasgrosman/wav2vec2-large-xlsr-53-persian  (183 clips)  [psrb]
  [100/183]  running avg WER = 0.5017

  jonatasgrosman/wav2vec2-large-xlsr-53-persian   [psrb]
  Corpus WER         : 0.4938   95% CI [0.4536, 0.5370]
  WER P50/P90/P95    : 0.474 / 0.850 / 0.937
  Corpus CER / chrF  : 0.1761 / 67.26
  SER / perfect / catastrophic : 0.973 / 2.7% / 41.0%
  Hallucinated / empty : 0.0% / 0.0%
  Sub/Ins/Del        : 0.3850 / 0.0282 / 0.0824
  RTF 0.0185  (54.05x real-time)   peak VRAM 5.33 GB
  Saved -> /kaggle/working/psrb/results__jonatasgrosman__wav2vec2-large-xlsr-53-persian.csv


---
## 4 · Meta SeamlessM4T

### 4.1 — hf-seamless-m4t-medium

In [31]:
MODEL_ID   = "facebook/hf-seamless-m4t-medium"
print(f"Loading {MODEL_ID} ...")
sm_proc  = AutoProcessor.from_pretrained(MODEL_ID)
sm_model = (
    SeamlessM4TModel.from_pretrained(MODEL_ID)
    .to(device).eval()
)
print("Ready.")

Loading facebook/hf-seamless-m4t-medium ...


preprocessor_config.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/4.85M [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/1371 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/4.84G [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to text_encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to text_decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie t2u_model.model.decoder.embed_tokens.weight to t2u_model.lm_head.weight, but both are present in the checkpoints, so we will NOT tie them

generation_config.json: 0.00B [00:00, ?B/s]

Ready.


Transcribe with `seamless-m4t-medium`, compute metrics, save CSV.

In [32]:
def _t(audio, sr):
    if sr != 16000:
        audio = torchaudio.functional.resample(
            torch.from_numpy(audio).unsqueeze(0), sr, 16000
        ).squeeze(0).numpy()
    inputs = sm_proc(audio=audio, sampling_rate=16000, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        out = sm_model.generate(**inputs, tgt_lang="pes", generate_speech=False)
    seqs = out.sequences if hasattr(out, "sequences") else out
    return sm_proc.tokenizer.batch_decode(seqs, skip_special_tokens=True)[0]

df7 = evaluate_model(_t, clips, "facebook/hf-seamless-m4t-medium")
del sm_model, sm_proc; free_gpu()


Evaluating: facebook/hf-seamless-m4t-medium  (183 clips)  [psrb]
  [100/183]  running avg WER = 0.4161

  facebook/hf-seamless-m4t-medium   [psrb]
  Corpus WER         : 0.3858   95% CI [0.3429, 0.4425]
  WER P50/P90/P95    : 0.316 / 0.772 / 1.000
  Corpus CER / chrF  : 0.1820 / 72.05
  SER / perfect / catastrophic : 0.967 / 3.3% / 25.7%
  Hallucinated / empty : 0.5% / 0.0%
  Sub/Ins/Del        : 0.2528 / 0.0474 / 0.0869
  RTF 0.0838  (11.93x real-time)   peak VRAM 12.88 GB
  Saved -> /kaggle/working/psrb/results__facebook__hf-seamless-m4t-medium.csv


### 4.2 — seamless-m4t-v2-large

In [33]:
MODEL_ID   = "facebook/seamless-m4t-v2-large"
print(f"Loading {MODEL_ID} ...")
sm_proc  = AutoProcessor.from_pretrained(MODEL_ID)
sm_model = (
    SeamlessM4Tv2Model.from_pretrained(MODEL_ID)
    .to(device).eval()
)
print("Ready.")

Loading facebook/seamless-m4t-v2-large ...


preprocessor_config.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/5.17M [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Instantiating a decoder SeamlessM4Tv2Attention without passing `layer_idx` is not recommended and will lead to errors during the forward call, if caching is used. Please make sure to provide a `layer_idx` when creating this class.


Loading weights:   0%|          | 0/2232 [00:00<?, ?it/s]

generation_config.json: 0.00B [00:00, ?B/s]

Ready.


Transcribe with `seamless-m4t-v2-large`, compute metrics, save CSV.

In [34]:
def _t(audio, sr):
    if sr != 16000:
        audio = torchaudio.functional.resample(
            torch.from_numpy(audio).unsqueeze(0), sr, 16000
        ).squeeze(0).numpy()
    inputs = sm_proc(audio=audio, sampling_rate=16000, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        out = sm_model.generate(**inputs, tgt_lang="pes", generate_speech=False)
    seqs = out.sequences if hasattr(out, "sequences") else out
    return sm_proc.tokenizer.batch_decode(seqs, skip_special_tokens=True)[0]

df8 = evaluate_model(_t, clips, "facebook/seamless-m4t-v2-large")
del sm_model, sm_proc; free_gpu()


Evaluating: facebook/seamless-m4t-v2-large  (183 clips)  [psrb]
  [100/183]  running avg WER = 0.3881

  facebook/seamless-m4t-v2-large   [psrb]
  Corpus WER         : 0.3916   95% CI [0.3531, 0.4347]
  WER P50/P90/P95    : 0.333 / 0.739 / 0.988
  Corpus CER / chrF  : 0.2040 / 69.29
  SER / perfect / catastrophic : 0.940 / 6.0% / 25.7%
  Hallucinated / empty : 0.0% / 0.0%
  Sub/Ins/Del        : 0.2713 / 0.0356 / 0.0851
  RTF 0.1928  (5.19x real-time)   peak VRAM 14.61 GB
  Saved -> /kaggle/working/psrb/results__facebook__seamless-m4t-v2-large.csv


---
## 5 · Meta MMS (CTC + Persian adapter)

### 5.1 — mms-1b-fl102

In [35]:
MODEL_ID = "facebook/mms-1b-fl102"
MMS_LANG = "fas"
print(f"Loading {MODEL_ID} ...")
mms_proc  = AutoProcessor.from_pretrained(MODEL_ID)
mms_model = Wav2Vec2ForCTC.from_pretrained(MODEL_ID).to(device).eval()
mms_proc.tokenizer.set_target_lang(MMS_LANG)
mms_model.load_adapter(MMS_LANG)
print(f"Ready (Persian adapter: {MMS_LANG}).")

Loading facebook/mms-1b-fl102 ...


preprocessor_config.json:   0%|          | 0.00/254 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/397 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/96.0 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/1096 [00:00<?, ?it/s]

adapter.fas.safetensors:   0%|          | 0.00/9.20M [00:00<?, ?B/s]

Ready (Persian adapter: fas).


Transcribe with `mms-1b-fl102`, compute metrics, save CSV.

In [36]:
def _t(audio, sr):
    if sr != 16000:
        audio = torchaudio.functional.resample(
            torch.from_numpy(audio).unsqueeze(0), sr, 16000
        ).squeeze(0).numpy()
    inputs = mms_proc(audio, sampling_rate=16000, return_tensors="pt")
    with torch.no_grad():
        logits = mms_model(inputs.input_values.to(device)).logits
    pred_ids = torch.argmax(logits, dim=-1)
    return mms_proc.decode(pred_ids[0])

df9 = evaluate_model(_t, clips, "facebook/mms-1b-fl102")
del mms_model, mms_proc; free_gpu()


Evaluating: facebook/mms-1b-fl102  (183 clips)  [psrb]
  [100/183]  running avg WER = 0.6472

  facebook/mms-1b-fl102   [psrb]
  Corpus WER         : 0.5518   95% CI [0.5186, 0.6012]
  WER P50/P90/P95    : 0.556 / 1.000 / 1.000
  Corpus CER / chrF  : 0.2091 / 63.05
  SER / perfect / catastrophic : 0.973 / 2.7% / 54.1%
  Hallucinated / empty : 1.6% / 0.0%
  Sub/Ins/Del        : 0.4203 / 0.0329 / 0.1046
  RTF 0.0481  (20.79x real-time)   peak VRAM 8.03 GB
  Saved -> /kaggle/working/psrb/results__facebook__mms-1b-fl102.csv


### 5.2 — mms-1b-all

In [37]:
MODEL_ID = "facebook/mms-1b-all"
MMS_LANG = "fas"
print(f"Loading {MODEL_ID} ...")
mms_proc  = AutoProcessor.from_pretrained(MODEL_ID)
mms_model = Wav2Vec2ForCTC.from_pretrained(MODEL_ID).to(device).eval()
mms_proc.tokenizer.set_target_lang(MMS_LANG)
mms_model.load_adapter(MMS_LANG)
print(f"Ready (Persian adapter: {MMS_LANG}).")

Loading facebook/mms-1b-all ...


preprocessor_config.json:   0%|          | 0.00/254 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/397 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/96.0 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/1096 [00:00<?, ?it/s]

adapter.fas.safetensors:   0%|          | 0.00/9.24M [00:00<?, ?B/s]

Ready (Persian adapter: fas).


Transcribe with `mms-1b-all`, compute metrics, save CSV.

In [38]:
def _t(audio, sr):
    if sr != 16000:
        audio = torchaudio.functional.resample(
            torch.from_numpy(audio).unsqueeze(0), sr, 16000
        ).squeeze(0).numpy()
    inputs = mms_proc(audio, sampling_rate=16000, return_tensors="pt")
    with torch.no_grad():
        logits = mms_model(inputs.input_values.to(device)).logits
    pred_ids = torch.argmax(logits, dim=-1)
    return mms_proc.decode(pred_ids[0])

df10 = evaluate_model(_t, clips, "facebook/mms-1b-all")
del mms_model, mms_proc; free_gpu()


Evaluating: facebook/mms-1b-all  (183 clips)  [psrb]
  [100/183]  running avg WER = 0.4756

  facebook/mms-1b-all   [psrb]
  Corpus WER         : 0.4733   95% CI [0.4365, 0.5143]
  WER P50/P90/P95    : 0.471 / 0.784 / 0.887
  Corpus CER / chrF  : 0.1665 / 68.13
  SER / perfect / catastrophic : 0.984 / 1.6% / 42.1%
  Hallucinated / empty : 0.0% / 0.0%
  Sub/Ins/Del        : 0.3569 / 0.0248 / 0.0935
  RTF 0.0479  (20.88x real-time)   peak VRAM 8.03 GB
  Saved -> /kaggle/working/psrb/results__facebook__mms-1b-all.csv


---
## Aggregate — master table for this round
Read this round's per-model CSVs, add BERTScore + semantic similarity, merge into one master table sorted by corpus WER, and (for the synthetic round) print WER-by-SNR.

In [ ]:
import glob
from bert_score import BERTScorer
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity as cos_sim

_seen, csv_files = set(), []
for _p in sorted(glob.glob(os.path.join(OUT_DIR, "results__*.csv")) +
                 glob.glob("/kaggle/input/**/results__*.csv", recursive=True)):
    if os.path.basename(_p) not in _seen:
        _seen.add(os.path.basename(_p)); csv_files.append(_p)
csv_files = sorted(csv_files)
print(f"[{DATASET}] Found {len(csv_files)} result file(s) (OUT_DIR + /kaggle/input)\n")

BERT_MODEL  = "HooshvareLab/bert-base-parsbert-uncased"
BERT_LAYERS = 9
print(f"Loading BERTScore model ({BERT_MODEL})...")
bert_scorer = BERTScorer(model_type=BERT_MODEL, num_layers=BERT_LAYERS, device=device, idf=False)
bert_scorer._tokenizer.model_max_length = 512
print("BERTScore ready.")
print("Loading sentence encoder (paraphrase-multilingual-MiniLM-L12-v2)...")
sem_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2", device=device)
print("Sentence encoder ready.\n")

rows, per_clip_all = [], []
for path in csv_files:
    df_tmp   = pd.read_csv(path)
    summary  = df_tmp[df_tmp["clip_id"].astype(str) == "SUMMARY"].iloc[0]
    per_clip = df_tmp[df_tmp["clip_id"].astype(str) != "SUMMARY"].copy()

    fname      = os.path.basename(path)
    model_name = fname[len("results__"):-len(".csv")].replace("__", "/", 1)
    print(f"── {model_name}")

    valid = per_clip[per_clip["reference"].fillna("").str.strip() != ""].copy()
    refs  = valid["reference"].fillna("").astype(str).tolist()
    hyps  = valid["hypothesis"].fillna("").astype(str).tolist()

    try:
        _, _, F1 = bert_scorer.score(hyps, refs, verbose=False)
        bertscore_f1 = round(float(F1.mean()), 4)
    except Exception as e:
        bertscore_f1 = float("nan"); print(f"   BERTScore FAILED ({e})")

    try:
        ref_embs = sem_model.encode(refs, batch_size=64, show_progress_bar=False)
        hyp_embs = sem_model.encode(hyps, batch_size=64, show_progress_bar=False)
        sims = [float(cos_sim([r], [h])[0][0]) for r, h in zip(ref_embs, hyp_embs)]
        avg_sem_sim = round(float(np.mean(sims)), 4)
    except Exception as e:
        avg_sem_sim = float("nan"); print(f"   Semantic sim FAILED ({e})")

    per_clip["model"] = model_name
    per_clip_all.append(per_clip)

    rows.append({
        "model"                      : model_name,
        "corpus_wer"                 : float(summary["wer"]),
        "wer_ci_low"                 : float(summary.get("wer_ci_low",  float("nan"))),
        "wer_ci_high"                : float(summary.get("wer_ci_high", float("nan"))),
        "corpus_cer"                 : float(summary["cer"]),
        "corpus_chrf"                : float(summary.get("chrf", float("nan"))),
        "bertscore_f1"               : bertscore_f1,
        "avg_semantic_similarity"    : avg_sem_sim,
        "ser"                        : float(summary.get("ser", float("nan"))),
        "wer_p50"                    : float(summary.get("wer_p50", float("nan"))),
        "wer_p90"                    : float(summary.get("wer_p90", float("nan"))),
        "wer_p95"                    : float(summary.get("wer_p95", float("nan"))),
        "pct_perfect"                : float(summary.get("pct_perfect", float("nan"))),
        "pct_catastrophic"           : float(summary.get("pct_catastrophic", float("nan"))),
        "pct_hallucinated"           : float(summary.get("is_hallucination", float("nan"))),
        "avg_delta_wer"              : float(summary.get("delta_wer", float("nan"))),
        "sub_rate"                   : float(summary.get("sub_rate", float("nan"))),
        "ins_rate"                   : float(summary.get("ins_rate", float("nan"))),
        "del_rate"                   : float(summary.get("del_rate", float("nan"))),
        "avg_hallucination_ratio"    : float(summary.get("hallucination_ratio", float("nan"))),
        "avg_script_contamination"   : float(summary.get("script_contamination", float("nan"))),
        "avg_repetition_score"       : float(summary.get("repetition_score", float("nan"))),
        "empty_rate"                 : float(summary.get("is_empty", float("nan"))),
        "avg_punct_f1"               : float(summary.get("punct_f1", float("nan"))),
        "avg_no_speech_prob"         : float(summary.get("no_speech_prob", float("nan"))),
        "avg_rtf"                    : float(summary["rtf"]),
        "throughput_audio_hrs_per_hr": float(summary.get("throughput_audio_hrs_per_hr", float("nan"))),
        "peak_vram_gb"               : float(summary.get("peak_vram_gb", float("nan"))),
        "wer_lt5s"                   : float(summary.get("wer_lt5s", float("nan"))),
        "wer_5to15s"                 : float(summary.get("wer_5to15s", float("nan"))),
        "wer_gt15s"                  : float(summary.get("wer_gt15s", float("nan"))),
        "avg_inf_time_sec"           : float(summary["inference_time_sec"]),
        "n_clips"                    : int(len(per_clip)),
    })

master = pd.DataFrame(rows).sort_values("corpus_wer").reset_index(drop=True)
master.index += 1
master.index.name = "rank"
master_path = os.path.join(OUT_DIR, "results_master.csv")
master.to_csv(master_path)
print(f"\n[{DATASET}] Master saved -> {master_path}\n")

display_cols = [
    "model", "corpus_wer", "wer_ci_low", "wer_ci_high",
    "wer_p50", "pct_catastrophic", "pct_hallucinated", "empty_rate",
    "throughput_audio_hrs_per_hr", "peak_vram_gb",
]
print(master[display_cols].to_string())

# ── WER-by-SNR breakdown (synthetic round) ───────────────────────
allpc = pd.concat(per_clip_all, ignore_index=True) if per_clip_all else pd.DataFrame()
if not allpc.empty and "snr_db" in allpc.columns and allpc["snr_db"].notna().any():
    allpc["snr_db"] = pd.to_numeric(allpc["snr_db"], errors="coerce")
    allpc["wer"]    = pd.to_numeric(allpc["wer"], errors="coerce")
    pivot = (allpc.dropna(subset=["snr_db"])
                  .pivot_table(index="model", columns="snr_db", values="wer", aggfunc="mean")
                  .round(4))
    print("\nWER by SNR (dB) — mean per-clip WER:")
    print(pivot.to_string())
    pivot.to_csv(os.path.join(OUT_DIR, "wer_by_snr.csv"))